# 🌟 LLFormer: Ultra-High-Definition Low-Light Image Enhancement
## Complete Baseline Reproduction from Epoch 0 on Google Colab
**Paper:** *LLFormer: Ultra-High-Definition Low-Light Image Enhancement* (AAAI 2023)  
**Authors:** Tao Wang, Kaihao Zhang, Tianrun Shen, Wenhan Luo, Bjorn Stenger, Tong Lu  
**Objective:** Complete baseline reproduction without pre-trained checkpoints (Training from scratch starting at Epoch 0 / 1).  
**Dataset:** LOL (Low-Light) Benchmark Dataset (485 training pairs, 15 evaluation pairs).

---
### 📌 Workflow Overview
1. **Environment Setup & Verification:** Hardware specs (GPU, VRAM), Python, PyTorch, CUDA, libraries, Git commit.
2. **Dataset Acquisition & Preprocessing Verification:** Download LOL dataset, check Low/Normal pairs, image dimensions, EDA brightness stats.
3. **Model Architecture & Initialization:** Clean LLFormer instance with random initialization (no pre-trained weights).
4. **Training from Scratch:** 50 epochs with PyTorch AMP FP16, Cosine Annealing scheduler, SmoothL1 loss, tracking logs per epoch.
5. **Validation & Evaluation:** Validation metrics (PSNR, SSIM, LPIPS) across epochs and per-image evaluation on LOL eval15.
6. **Automated Paper Assets Export:** Automatic generation of publication-ready LaTeX tables (.tex), CSV summaries, and IEEE-format high-res figures (.png) for direct paper drafting.


---
## Section 1: Environment Setup & Hardware Diagnostics
We mount Google Drive to safeguard all checkpoints, logs, and generated paper assets against Colab session disconnects.


In [ ]:
# Mount Google Drive for persistent storage
import os
import sys

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/LLFormer_Colab_Reproduction'
except Exception as e:
    print("Not running in Colab or Drive mount skipped. Using local directory.")
    DRIVE_DIR = './colab_reproduction_drive'

OUTPUT_DIR = os.path.join(DRIVE_DIR, 'paper_output')
CHECKPOINT_DIR = os.path.join(DRIVE_DIR, 'checkpoints')
LOG_DIR = os.path.join(DRIVE_DIR, 'logs')

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

print(f"Persistent Storage Directory : {DRIVE_DIR}")
print(f"Paper Assets Export Directory : {OUTPUT_DIR}")


In [ ]:
# Install required Python dependencies
!pip install -q einops pytorch-msssim scikit-image lpips timm gdown tqdm matplotlib seaborn


In [ ]:
# Clone LLFormer repository if not present
import os
import subprocess

if not os.path.exists('/content/LLFormer'):
    !git clone https://github.com/TaoWangzj/LLFormer.git /content/LLFormer
    %cd /content/LLFormer
else:
    %cd /content/LLFormer

print("Current Working Directory:", os.getcwd())


In [ ]:
# Hardware & Software Diagnostics
import sys
import platform
import json
import torch
import numpy as np
import einops
import skimage
import lpips
import torchvision

cuda_available = torch.cuda.is_available()
device = torch.device('cuda' if cuda_available else 'cpu')

env_info = {
    "Operating_System": platform.platform(),
    "Python_Version": sys.version.split()[0],
    "PyTorch_Version": torch.__version__,
    "CUDA_Available": cuda_available,
    "CUDA_Version": torch.version.cuda if cuda_available else "N/A",
    "GPU_Name": torch.cuda.get_device_name(0) if cuda_available else "N/A",
    "GPU_VRAM_GB": round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2) if cuda_available else 0.0,
    "einops_version": einops.__version__,
    "skimage_version": skimage.__version__,
    "torchvision_version": torchvision.__version__,
}

try:
    commit_hash = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd='/content/LLFormer').decode().strip()
    env_info["Git_Commit"] = commit_hash[:8]
except Exception:
    env_info["Git_Commit"] = "Official_Repo"

print("=" * 65)
print("              SYSTEM & ENVIRONMENT SPECIFICATIONS")
print("=" * 65)
for k, v in env_info.items():
    print(f"  {k:<22}: {v}")
print("=" * 65)

# Save environment specifications for paper Table
with open(os.path.join(OUTPUT_DIR, 'environment_specs.json'), 'w') as f:
    json.dump(env_info, f, indent=4)
print(f"Saved: {os.path.join(OUTPUT_DIR, 'environment_specs.json')}")


---
## Section 2: Dataset Acquisition, Integrity Verification & EDA
We download the benchmark **LOL dataset** (`our485` for training, `eval15` for evaluation) and rigorously inspect image pairing, dimensions, and brightness distributions.


In [ ]:
# Download & Setup LOL Dataset
import os
import shutil

dataset_dir = '/content/LLFormer/LOLdataset'
os.makedirs(dataset_dir, exist_ok=True)

# Function to auto-flatten nested folders if any
def flatten_nested(parent_dir):
    for sub in ['lol_dataset', 'LOLdataset']:
        nested = os.path.join(parent_dir, sub)
        if os.path.exists(nested):
            for item in os.listdir(nested):
                src = os.path.join(nested, item)
                dst = os.path.join(parent_dir, item)
                if not os.path.exists(dst):
                    shutil.move(src, dst)
            shutil.rmtree(nested, ignore_errors=True)

flatten_nested(dataset_dir)
flatten_nested('/content/LLFormer')

# Check if dataset is already properly in place
if not os.path.exists(os.path.join(dataset_dir, 'our485', 'low')):
    print("Downloading LOL dataset from Hugging Face direct mirror...")
    !wget -q --show-progress -O /content/lol_dataset.zip "https://huggingface.co/datasets/geekyrakshit/LoL-Dataset/resolve/main/lol_dataset.zip"
    print("Extracting dataset...")
    !unzip -q -o /content/lol_dataset.zip -d /content/LLFormer/LOLdataset/
    flatten_nested(dataset_dir)
    !rm -rf /content/LLFormer/LOLdataset/__MACOSX

assert os.path.exists(os.path.join(dataset_dir, 'our485', 'low')), "Error: our485/low still not found!"
train_cnt = len(os.listdir(os.path.join(dataset_dir, 'our485', 'low')))
eval_cnt = len(os.listdir(os.path.join(dataset_dir, 'eval15', 'low')))
print("=" * 65)
print("             LOL DATASET SETUP COMPLETED SUCCESSFULLY")
print("=" * 65)
print(f"  Dataset Directory : {dataset_dir}")
print(f"  Training Pairs    : {train_cnt} in our485/low")
print(f"  Evaluation Pairs  : {eval_cnt} in eval15/low")
print("=" * 65)


In [ ]:
# Verification of Low-Light / Normal-Light Image Pairing and EDA Analysis
from PIL import Image
import glob
import pandas as pd

train_low_dir = os.path.join(dataset_dir, 'our485', 'low')
train_high_dir = os.path.join(dataset_dir, 'our485', 'high')
val_low_dir = os.path.join(dataset_dir, 'eval15', 'low')
val_high_dir = os.path.join(dataset_dir, 'eval15', 'high')

train_low_files = sorted([f for f in os.listdir(train_low_dir) if f.endswith(('.png', '.jpg'))])
train_high_files = sorted([f for f in os.listdir(train_high_dir) if f.endswith(('.png', '.jpg'))])
val_low_files = sorted([f for f in os.listdir(val_low_dir) if f.endswith(('.png', '.jpg'))])
val_high_files = sorted([f for f in os.listdir(val_high_dir) if f.endswith(('.png', '.jpg'))])

assert train_low_files == train_high_files, "Mismatch in Training low/high image filenames!"
assert val_low_files == val_high_files, "Mismatch in Validation low/high image filenames!"

# Compute EDA metrics across datasets
def compute_eda(file_list, low_dir, high_dir):
    stats = []
    for fname in file_list:
        p_low = os.path.join(low_dir, fname)
        p_high = os.path.join(high_dir, fname)
        im_l = np.array(Image.open(p_low).convert('RGB'))
        im_h = np.array(Image.open(p_high).convert('RGB'))
        stats.append({
            'filename': fname,
            'height': im_l.shape[0],
            'width': im_l.shape[1],
            'low_mean': np.mean(im_l),
            'high_mean': np.mean(im_h),
            'low_std': np.std(im_l),
            'high_std': np.std(im_h),
            'contrast_ratio': np.mean(im_h) / (np.mean(im_l) + 1e-5)
        })
    return pd.DataFrame(stats)

df_train = compute_eda(train_low_files, train_low_dir, train_high_dir)
df_val = compute_eda(val_low_files, val_low_dir, val_high_dir)

dataset_eda_summary = {
    "Train_Pairs_Count": len(df_train),
    "Val_Pairs_Count": len(df_val),
    "Image_Resolution": f"{df_train['width'].iloc[0]}x{df_train['height'].iloc[0]}",
    "Train_Low_Mean_Brightness": round(float(df_train['low_mean'].mean()), 2),
    "Train_High_Mean_Brightness": round(float(df_train['high_mean'].mean()), 2),
    "Val_Low_Mean_Brightness": round(float(df_val['low_mean'].mean()), 2),
    "Val_High_Mean_Brightness": round(float(df_val['high_mean'].mean()), 2),
    "Pairing_Integrity": "100% Validated"
}

print("=" * 65)
print("                    LOL DATASET EDA SUMMARY")
print("=" * 65)
for k, v in dataset_eda_summary.items():
    print(f"  {k:<28}: {v}")
print("=" * 65)

with open(os.path.join(OUTPUT_DIR, 'dataset_eda_summary.json'), 'w') as f:
    json.dump(dataset_eda_summary, f, indent=4)
df_val.to_csv(os.path.join(OUTPUT_DIR, 'eval15_image_properties.csv'), index=False)
print("Saved dataset EDA summaries.")


---
## Section 3: Model Architecture Configuration & Random Initialization
We instantiate the full **LLFormer** model with standard hyperparameters and randomly initialized weights. No pre-trained weights or transfer checkpoints are loaded.


In [ ]:
# Instantiate LLFormer from Scratch
import torch
import torch.nn as nn
from model.LLFormer import LLFormer

# Hyperparameters conforming to LLFormer baseline
model_config = {
    "inp_channels": 3,
    "out_channels": 3,
    "dim": 16,
    "num_blocks": [2, 4, 8, 16],
    "num_refinement_blocks": 2,
    "heads": [1, 2, 4, 8],
    "ffn_expansion_factor": 2.66,
    "bias": False,
    "LayerNorm_type": 'WithBias',
    "attention": True,
    "skip": False
}

model = LLFormer(**model_config).to(device)

# Verify parameter counts
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("=" * 65)
print("             LLFORMER ARCHITECTURE & PARAMETER COUNT")
print("=" * 65)
print(f"  Total Parameters     : {total_params:,} ({total_params/1e6:.2f} M)")
print(f"  Trainable Parameters : {trainable_params:,} ({trainable_params/1e6:.2f} M)")
print(f"  Attention Enabled    : {model_config['attention']}")
print(f"  FFN Expansion Factor : {model_config['ffn_expansion_factor']}")
print(f"  Block Hierarchy      : {model_config['num_blocks']} + Refinement: {model_config['num_refinement_blocks']}")
print("=" * 65)

with open(os.path.join(OUTPUT_DIR, 'model_architecture_config.json'), 'w') as f:
    json.dump({**model_config, "total_params": total_params, "trainable_params": trainable_params}, f, indent=4)


---
## Section 4: Training Pipeline from Epoch 0 (Full Training Loop)
We execute the complete 50-epoch baseline training on Google Colab with:
- **Optimizer:** Adam ($eta_1=0.9, eta_2=0.999, \epsilon=10^{-8}$)
- **Learning Rate:** $1.5 	imes 10^{-4}$ with **CosineAnnealingLR** decay down to $10^{-6}$
- **Objective Function:** Smooth L1 Loss ($\mathcal{L}_1$)
- **Data Augmentation:** Random $128 	imes 128$ crop, horizontal/vertical flips, $90^\circ$ rotations
- **Acceleration:** PyTorch Automatic Mixed Precision (AMP FP16)
- **Checkpoints & Logs:** Real-time logging of loss, validation PSNR, SSIM, and automatic saving of `model_bestPSNR.pth` and `model_latest.pth`.


In [ ]:
# Full Training from Epoch 0
import time
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from transform.data_RGB import get_training_data, get_validation_data
import utils

# Training Hyperparameters
EPOCHS = 50
BATCH_SIZE = 4
TRAIN_PATCH_SIZE = 128
VAL_PATCH_SIZE = 128
INITIAL_LR = 1.5e-4
MIN_LR = 1e-6

train_hyperparams = {
    "Epochs": EPOCHS,
    "Batch_Size": BATCH_SIZE,
    "Train_Patch_Size": f"{TRAIN_PATCH_SIZE}x{TRAIN_PATCH_SIZE}",
    "Val_Patch_Size": f"{VAL_PATCH_SIZE}x{VAL_PATCH_SIZE}",
    "Optimizer": "Adam (beta1=0.9, beta2=0.999)",
    "Loss_Function": "SmoothL1Loss",
    "Learning_Rate_Initial": INITIAL_LR,
    "Learning_Rate_Min": MIN_LR,
    "Scheduler": "CosineAnnealingLR",
    "Precision": "Mixed Precision (AMP FP16)",
    "Training_From_Scratch": True
}

with open(os.path.join(OUTPUT_DIR, 'training_hyperparameters.json'), 'w') as f:
    json.dump(train_hyperparams, f, indent=4)

# Data Loaders
train_dataset = get_training_data(os.path.join(dataset_dir, 'our485'), {'patch_size': TRAIN_PATCH_SIZE})
train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)

val_dataset = get_validation_data(os.path.join(dataset_dir, 'eval15'), {'patch_size': VAL_PATCH_SIZE})
val_loader = DataLoader(dataset=val_dataset, batch_size=1, shuffle=False, num_workers=0, pin_memory=True)

optimizer = optim.Adam(model.parameters(), lr=INITIAL_LR, betas=(0.9, 0.999), eps=1e-8)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=MIN_LR)
criterion = nn.SmoothL1Loss()
scaler = torch.amp.GradScaler('cuda') if cuda_available else None

# Training History Containers
history = {
    'epoch': [],
    'train_loss': [],
    'val_psnr': [],
    'val_ssim': [],
    'lr': [],
    'epoch_time_seconds': []
}

best_val_psnr = -1.0
total_train_start_time = time.time()

print(f"Starting Training: 1 to {EPOCHS} Epochs from Scratch...")
print("=" * 75)

for epoch in range(1, EPOCHS + 1):
    epoch_start_time = time.time()
    model.train()
    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch:02d}/{EPOCHS:02d}", leave=False)
    for data in pbar:
        target = data[0].to(device)
        input_ = data[1].to(device)

        optimizer.zero_grad(set_to_none=True)

        if cuda_available:
            with torch.amp.autocast('cuda'):
                restored = model(input_)
                loss = criterion(restored, target)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            restored = model(input_)
            loss = criterion(restored, target)
            loss.backward()
            optimizer.step()

        running_loss += loss.item()
        pbar.set_postfix({'loss': f"{loss.item():.4f}"})

    epoch_loss = running_loss / len(train_loader)

    # Validation Phase
    model.eval()
    epoch_psnrs, epoch_ssims = [], []
    with torch.no_grad():
        for data_val in val_loader:
            target = data_val[0].to(device)
            input_ = data_val[1].to(device)
            h, w = target.shape[2], target.shape[3]

            if cuda_available:
                with torch.amp.autocast('cuda'):
                    restored = model(input_)
            else:
                restored = model(input_)

            restored = torch.clamp(restored[:, :, :h, :w].float(), 0, 1)
            epoch_psnrs.append(utils.torchPSNR(restored[0], target[0]).item())
            epoch_ssims.append(utils.torchSSIM(restored, target).item())

    val_psnr = float(np.mean(epoch_psnrs))
    val_ssim = float(np.mean(epoch_ssims))
    current_lr = scheduler.get_last_lr()[0]
    scheduler.step()
    epoch_duration = time.time() - epoch_start_time

    # Record History
    history['epoch'].append(epoch)
    history['train_loss'].append(round(epoch_loss, 5))
    history['val_psnr'].append(round(val_psnr, 4))
    history['val_ssim'].append(round(val_ssim, 4))
    history['lr'].append(current_lr)
    history['epoch_time_seconds'].append(round(epoch_duration, 2))

    log_msg = f"[Epoch {epoch:02d}/{EPOCHS:02d}] Loss: {epoch_loss:.4f} | Val PSNR: {val_psnr:.2f} dB | Val SSIM: {val_ssim:.4f} | LR: {current_lr:.6f} | Time: {epoch_duration:.1f}s"
    print(log_msg)

    # Save Best Model
    if val_psnr > best_val_psnr:
        best_val_psnr = val_psnr
        best_ckpt = {
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'val_psnr': val_psnr,
            'val_ssim': val_ssim,
            'config': model_config
        }
        torch.save(best_ckpt, os.path.join(CHECKPOINT_DIR, 'model_bestPSNR.pth'))
        print(f"  >>> Checkpoint Saved: Best PSNR = {best_val_psnr:.2f} dB at Epoch {epoch} <<<")

    # Save Latest Model
    latest_ckpt = {
        'epoch': epoch,
        'state_dict': model.state_dict(),
        'optimizer': optimizer.state_dict(),
        'val_psnr': val_psnr,
        'val_ssim': val_ssim,
        'config': model_config
    }
    torch.save(latest_ckpt, os.path.join(CHECKPOINT_DIR, 'model_latest.pth'))

    # Save JSON training progress
    with open(os.path.join(LOG_DIR, 'training_progress.json'), 'w') as f:
        json.dump(history, f, indent=4)

total_training_time = time.time() - total_train_start_time
print("=" * 75)
print(f"Training Complete! Total Duration: {total_training_time/60:.2f} minutes ({total_training_time/3600:.2f} hours)")
print(f"Best Validation PSNR: {best_val_psnr:.2f} dB")


---
## Section 5: Full Evaluation on Evaluation Set (Full Resolution & LPIPS)
We load the best-performing checkpoint (`model_bestPSNR.pth`) and evaluate on the full-resolution $400 	imes 600$ images of the LOL evaluation set (`eval15`), calculating per-image **PSNR, SSIM, and LPIPS** (AlexNet perceptual distance).


In [ ]:
# Rigorous Evaluation on LOL eval15 with Best Model
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from skimage.metrics import structural_similarity as ssim_fn
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
import lpips

# Load Best Checkpoint
best_weights_path = os.path.join(CHECKPOINT_DIR, 'model_bestPSNR.pth')
checkpoint = torch.load(best_weights_path, map_location=device)
model.load_state_dict(checkpoint['state_dict'])
model.eval()

# Perceptual metric LPIPS
loss_fn_alex = lpips.LPIPS(net='alex').to(device)

eval_results = []
os.makedirs(os.path.join(OUTPUT_DIR, 'enhanced_images'), exist_ok=True)

print(f"Loaded Best Model from Epoch {checkpoint['epoch']} (Val PSNR: {checkpoint['val_psnr']:.2f} dB)")
print("Evaluating on LOL eval15 test images...")

with torch.no_grad():
    for fname in val_low_files:
        p_low = os.path.join(val_low_dir, fname)
        p_high = os.path.join(val_high_dir, fname)

        raw_low = Image.open(p_low).convert('RGB')
        raw_high = Image.open(p_high).convert('RGB')

        low_tensor = TF.to_tensor(raw_low).unsqueeze(0).to(device)
        high_tensor = TF.to_tensor(raw_high).unsqueeze(0).to(device)

        h, w = low_tensor.shape[2], low_tensor.shape[3]
        mul = 16
        H, W = ((h + mul - 1) // mul) * mul, ((w + mul - 1) // mul) * mul
        padh, padw = H - h, W - w
        inp_padded = F.pad(low_tensor, (0, padw, 0, padh), 'reflect')

        if cuda_available:
            with torch.amp.autocast('cuda'):
                pred = model(inp_padded)
        else:
            pred = model(inp_padded)

        pred = torch.clamp(pred[:, :, :h, :w].float(), 0, 1)

        # Compute LPIPS
        # Scale to [-1, 1] for LPIPS
        lp_score = loss_fn_alex(pred * 2.0 - 1.0, high_tensor * 2.0 - 1.0).item()

        img_gt = np.array(raw_high)
        img_low = np.array(raw_low)
        img_pred = (pred[0].cpu().permute(1, 2, 0).numpy() * 255.0).round().astype(np.uint8)

        # Metrics for LLFormer
        m_psnr = psnr_fn(img_gt, img_pred, data_range=255)
        m_ssim = ssim_fn(img_gt, img_pred, channel_axis=2, data_range=255)

        # Baseline metrics for original Low-Light Input
        base_psnr = psnr_fn(img_gt, img_low, data_range=255)
        base_ssim = ssim_fn(img_gt, img_low, channel_axis=2, data_range=255)

        # Save enhanced image
        Image.fromarray(img_pred).save(os.path.join(OUTPUT_DIR, 'enhanced_images', fname))

        eval_results.append({
            'Image': fname,
            'Low_PSNR': round(base_psnr, 2),
            'Low_SSIM': round(base_ssim, 4),
            'LLFormer_PSNR': round(m_psnr, 2),
            'LLFormer_SSIM': round(m_ssim, 4),
            'LLFormer_LPIPS': round(lp_score, 4),
            'Delta_PSNR': round(m_psnr - base_psnr, 2),
            'Delta_SSIM': round(m_ssim - base_ssim, 4)
        })

df_eval = pd.DataFrame(eval_results)
df_eval.to_csv(os.path.join(OUTPUT_DIR, 'per_image_eval_results.csv'), index=False)

mean_llformer_psnr = df_eval['LLFormer_PSNR'].mean()
mean_llformer_ssim = df_eval['LLFormer_SSIM'].mean()
mean_llformer_lpips = df_eval['LLFormer_LPIPS'].mean()
mean_base_psnr = df_eval['Low_PSNR'].mean()
mean_base_ssim = df_eval['Low_SSIM'].mean()

print("=" * 65)
print("             LOL EVAL15 FINAL TEST SET EVALUATION")
print("=" * 65)
print(f"  Unprocessed Input : PSNR = {mean_base_psnr:.2f} dB | SSIM = {mean_base_ssim:.4f}")
print(f"  LLFormer (Scratch): PSNR = {mean_llformer_psnr:.2f} dB | SSIM = {mean_llformer_ssim:.4f} | LPIPS = {mean_llformer_lpips:.4f}")
print(f"  Overall Gain      : +{mean_llformer_psnr - mean_base_psnr:.2f} dB PSNR | +{mean_llformer_ssim - mean_base_ssim:.4f} SSIM")
print("=" * 65)


---
## Section 6: Automated Paper Figures & Tables Generation
In this final section, we automatically compile and export:
1. **Convergence Curves:** Loss, PSNR, SSIM, and Learning Rate curves over 50 epochs.
2. **Per-Image Performance Bar Chart:** Side-by-side comparison of baseline vs LLFormer across all 15 test scenes.
3. **High-Resolution Visual Comparison Gallery:** Low-Light Input vs LLFormer vs Ground Truth with zoom patches.
4. **Brightness Histogram & Residual Error Analysis:** Distribution shifting and spatial error maps $|\text{LLFormer} - \text{GT}|$.
5. **Publication-Ready LaTeX Tables:** Complete `.tex` files for Tables 1, 2, and 3 directly formatted for IEEE/AAAI templates.


In [ ]:
# Figure 1: Training Convergence Curves (Loss, PSNR, SSIM, LR)
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 2, figsize=(14, 10), facecolor='#ffffff')
epochs_arr = history['epoch']

# Subplot 1: Training Loss
axes[0, 0].plot(epochs_arr, history['train_loss'], color='#0284c7', linewidth=2.2, marker='o', markersize=3, label='Smooth L1 Loss')
axes[0, 0].set_title('Training Loss Convergence', fontsize=12, fontweight='bold', pad=8)
axes[0, 0].set_xlabel('Epoch', fontsize=10)
axes[0, 0].set_ylabel('Loss', fontsize=10)
axes[0, 0].grid(True, linestyle='--', alpha=0.5)
axes[0, 0].legend()

# Subplot 2: Validation PSNR
axes[0, 1].plot(epochs_arr, history['val_psnr'], color='#10b981', linewidth=2.2, marker='s', markersize=3, label='Val PSNR (dB)')
axes[0, 1].axhline(y=mean_llformer_psnr, color='#f59e0b', linestyle=':', label=f'Best Test: {mean_llformer_psnr:.2f} dB')
axes[0, 1].set_title('Validation PSNR Metric', fontsize=12, fontweight='bold', pad=8)
axes[0, 1].set_xlabel('Epoch', fontsize=10)
axes[0, 1].set_ylabel('PSNR (dB)', fontsize=10)
axes[0, 1].grid(True, linestyle='--', alpha=0.5)
axes[0, 1].legend()

# Subplot 3: Validation SSIM
axes[1, 0].plot(epochs_arr, history['val_ssim'], color='#8b5cf6', linewidth=2.2, marker='^', markersize=3, label='Val SSIM')
axes[1, 0].set_title('Validation Structural Similarity (SSIM)', fontsize=12, fontweight='bold', pad=8)
axes[1, 0].set_xlabel('Epoch', fontsize=10)
axes[1, 0].set_ylabel('SSIM', fontsize=10)
axes[1, 0].grid(True, linestyle='--', alpha=0.5)
axes[1, 0].legend()

# Subplot 4: Learning Rate Schedule
axes[1, 1].plot(epochs_arr, history['lr'], color='#ef4444', linewidth=2.0, label='CosineAnnealingLR')
axes[1, 1].set_title('Learning Rate Schedule', fontsize=12, fontweight='bold', pad=8)
axes[1, 1].set_xlabel('Epoch', fontsize=10)
axes[1, 1].set_ylabel('Learning Rate', fontsize=10)
axes[1, 1].grid(True, linestyle='--', alpha=0.5)
axes[1, 1].legend()

plt.suptitle('Figure 1: LLFormer Training Convergence & Validation Trajectory (50 Epochs, LOL Dataset)', fontsize=14, fontweight='bold', y=0.98)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
f1_path = os.path.join(OUTPUT_DIR, 'fig_training_curves.png')
plt.savefig(f1_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Generated Figure 1: {f1_path}")


In [ ]:
# Figure 2: Per-Image PSNR and SSIM Comparison on LOL eval15
fig, ax = plt.subplots(figsize=(14, 6), facecolor='#ffffff')
x = np.arange(len(df_eval))
width = 0.38

rects1 = ax.bar(x - width/2, df_eval['Low_PSNR'], width, label='Unprocessed Low-Light Input', color='#94a3b8', edgecolor='#64748b')
rects2 = ax.bar(x + width/2, df_eval['LLFormer_PSNR'], width, label='LLFormer (Reproduced Baseline)', color='#0284c7', edgecolor='#0369a1')

ax.set_ylabel('PSNR (dB)', fontsize=11, fontweight='bold')
ax.set_title('Figure 2: Per-Image Peak Signal-to-Noise Ratio (PSNR) across LOL eval15 Benchmark Images', fontsize=13, fontweight='bold', pad=12)
ax.set_xticks(x)
ax.set_xticklabels(df_eval['Image'], rotation=45, ha='right', fontsize=9)
ax.axhline(mean_llformer_psnr, color='#10b981', linestyle='--', linewidth=1.5, label=f'Mean LLFormer PSNR: {mean_llformer_psnr:.2f} dB')
ax.axhline(mean_base_psnr, color='#ef4444', linestyle=':', linewidth=1.5, label=f'Mean Input PSNR: {mean_base_psnr:.2f} dB')
ax.legend(loc='upper right', frameon=True, fontsize=10)
ax.grid(True, axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
f2_path = os.path.join(OUTPUT_DIR, 'fig_per_image_psnr.png')
plt.savefig(f2_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Generated Figure 2: {f2_path}")


In [ ]:
# Figure 3: High-Resolution Qualitative Comparisons with Zoom Boxes & Residuals
import matplotlib.patches as patches

sample_files = ['1.png', '22.png', '669.png', '778.png']
sample_files = [f for f in sample_files if f in val_low_files]
if len(sample_files) < 3:
    sample_files = val_low_files[:3]

fig, axes = plt.subplots(len(sample_files), 4, figsize=(18, 4.5 * len(sample_files)), facecolor='#ffffff')

for idx, fname in enumerate(sample_files):
    img_low = np.array(Image.open(os.path.join(val_low_dir, fname)).convert('RGB'))
    img_pred = np.array(Image.open(os.path.join(OUTPUT_DIR, 'enhanced_images', fname)).convert('RGB'))
    img_gt = np.array(Image.open(os.path.join(val_high_dir, fname)).convert('RGB'))
    
    diff_map = np.mean(np.abs(img_gt.astype(float) - img_pred.astype(float)), axis=2)
    
    row_psnr = df_eval.loc[df_eval['Image'] == fname, 'LLFormer_PSNR'].values[0]
    row_ssim = df_eval.loc[df_eval['Image'] == fname, 'LLFormer_SSIM'].values[0]

    # Col 1: Low-Light Input
    axes[idx, 0].imshow(img_low)
    axes[idx, 0].set_title(f"Input: {fname}", fontsize=11, fontweight='bold')
    axes[idx, 0].axis('off')

    # Col 2: LLFormer Prediction
    axes[idx, 1].imshow(img_pred)
    axes[idx, 1].set_title(f"LLFormer: {row_psnr:.2f} dB / {row_ssim:.4f}", fontsize=11, fontweight='bold', color='#0284c7')
    axes[idx, 1].axis('off')

    # Col 3: Ground Truth
    axes[idx, 2].imshow(img_gt)
    axes[idx, 2].set_title("Ground Truth", fontsize=11, fontweight='bold', color='#10b981')
    axes[idx, 2].axis('off')

    # Col 4: Residual Error Map
    im_err = axes[idx, 3].imshow(diff_map, cmap='inferno', vmin=0, vmax=100)
    axes[idx, 3].set_title(f"Error Map |Pred - GT|", fontsize=11, fontweight='bold', color='#ef4444')
    axes[idx, 3].axis('off')

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(im_err, cax=cbar_ax)

plt.suptitle('Figure 3: Qualitative Enhancement Results and Spatial Residual Error Distribution', fontsize=14, fontweight='bold', y=0.99)
plt.tight_layout(rect=[0, 0, 0.91, 0.98])
f3_path = os.path.join(OUTPUT_DIR, 'fig_visual_comparison_gallery.png')
plt.savefig(f3_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Generated Figure 3: {f3_path}")


In [ ]:
# Figure 4: Brightness Distribution Shift Analysis (Histogram)
target_sample = '1.png' if '1.png' in val_low_files else val_low_files[0]
img_l = np.array(Image.open(os.path.join(val_low_dir, target_sample)).convert('RGB'))
img_p = np.array(Image.open(os.path.join(OUTPUT_DIR, 'enhanced_images', target_sample)).convert('RGB'))
img_g = np.array(Image.open(os.path.join(val_high_dir, target_sample)).convert('RGB'))

fig, ax = plt.subplots(figsize=(10, 5), facecolor='#ffffff')
ax.hist(np.mean(img_l, axis=2).ravel(), bins=60, color='#94a3b8', alpha=0.5, density=True, label='Low-Light Input')
ax.hist(np.mean(img_p, axis=2).ravel(), bins=60, color='#0284c7', alpha=0.5, density=True, label='LLFormer Restored')
ax.hist(np.mean(img_g, axis=2).ravel(), bins=60, color='#10b981', alpha=0.4, histtype='step', linewidth=2, label='Ground Truth Target')

ax.set_title(f'Figure 4: Pixel Luminance Distribution Shift Analysis ({target_sample})', fontsize=12, fontweight='bold', pad=10)
ax.set_xlabel('Luminance Intensity (0 - 255)', fontsize=10)
ax.set_ylabel('Probability Density', fontsize=10)
ax.legend(frameon=True, fontsize=10)
ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
f4_path = os.path.join(OUTPUT_DIR, 'fig_histogram_analysis.png')
plt.savefig(f4_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"Generated Figure 4: {f4_path}")


In [ ]:
# Export Automated LaTeX Tables (.tex) for Paper Writing

# Table 1: Per-Image Evaluation on LOL eval15
tex_table1 = r"""\begin{table}[htbp]
\centering
\caption{Per-Image Quantitative Evaluation on LOL eval15 Dataset}
\label{tab:per_image_eval}
\begin{tabular}{lcccccc}
\hline
\textbf{Image} & \multicolumn{2}{c}{\textbf{Low-Light Input}} & \multicolumn{3}{c}{\textbf{LLFormer (Reproduced)}} & \textbf{Gain} \\
 & PSNR (dB) & SSIM & PSNR (dB) & SSIM & LPIPS & $\Delta$PSNR \\
\hline
"""

for _, row in df_eval.iterrows():
    tex_table1 += f"{row['Image']} & {row['Low_PSNR']:.2f} & {row['Low_SSIM']:.4f} & {row['LLFormer_PSNR']:.2f} & {row['LLFormer_SSIM']:.4f} & {row['LLFormer_LPIPS']:.4f} & +{row['Delta_PSNR']:.2f} \\\\
"

tex_table1 += r"""\hline
\textbf{Mean} & \textbf{""" + f"{mean_base_psnr:.2f}" + r"""} & \textbf{""" + f"{mean_base_ssim:.4f}" + r"""} & \textbf{""" + f"{mean_llformer_psnr:.2f}" + r"""} & \textbf{""" + f"{mean_llformer_ssim:.4f}" + r"""} & \textbf{""" + f"{mean_llformer_lpips:.4f}" + r"""} & \textbf{+""" + f"{mean_llformer_psnr - mean_base_psnr:.2f}" + r"""} \\
\hline
\end{tabular}
\end{table}
"""

with open(os.path.join(OUTPUT_DIR, 'table1_per_image_evaluation.tex'), 'w') as f:
    f.write(tex_table1)

# Table 2: Comparative Evaluation against Prior Methods (Benchmark Table)
# Note: Prior works results from peer-reviewed literature (BMVC 2018, CVPR 2019, ICCV 2021, CVPR 2022, AAAI 2023)
tex_table2 = f"""\begin{{table}}[htbp]
\centering
\caption{{Quantitative Comparison on the LOL Benchmark Dataset}}
\label{{tab:benchmark_comparison}}
\begin{{tabular}}{{lcccc}}
\hline
\textbf{{Method}} & \textbf{{Venue / Year}} & \textbf{{PSNR (dB)}}$\uparrow$ & \textbf{{SSIM}}$\uparrow$ & \textbf{{LPIPS}}$\downarrow$ \\
\hline
Low-Light Input & --- & {mean_base_psnr:.2f} & {mean_base_ssim:.4f} & --- \\
RetinexNet (Wei et al.) & BMVC'18 & 16.77 & 0.560 & 0.388 \\
MBLLEN (Lv et al.) & BMVC'18 & 17.90 & 0.724 & 0.247 \\
KinD (Zhang et al.) & ACM MM'19 & 20.87 & 0.800 & 0.178 \\
EnlightenGAN (Jiang et al.) & IEEE TIP'21 & 17.48 & 0.650 & 0.322 \\
Restormer (Zamir et al.) & CVPR'22 & 22.43 & 0.823 & 0.147 \\
LLFormer (Official Paper) & AAAI'23 & 23.65 & 0.816 & 0.160 \\
\hline
\textbf{{LLFormer (Our Reproduction)}} & \textbf{{Baseline}} & \textbf{{{mean_llformer_psnr:.2f}}} & \textbf{{{mean_llformer_ssim:.4f}}} & \textbf{{{mean_llformer_lpips:.4f}}} \\
\hline
\end{{tabular}}
\end{{table}}
"""

with open(os.path.join(OUTPUT_DIR, 'table2_benchmark_comparison.tex'), 'w') as f:
    f.write(tex_table2)

# Table 3: Experimental Configuration & Training Specifications
tex_table3 = f"""\begin{{table}}[htbp]
\centering
\caption{{Hardware, Software, and Hyperparameter Specifications}}
\label{{tab:training_specifications}}
\begin{{tabular}}{{ll}}
\hline
\textbf{{Parameter / Environment Component}} & \textbf{{Specification / Value}} \\
\hline
Hardware Accelerator & {env_info.get('GPU_Name', 'N/A')} ({env_info.get('GPU_VRAM_GB', '0')} GB VRAM) \\
Deep Learning Framework & PyTorch {env_info.get('PyTorch_Version', 'N/A')} + CUDA {env_info.get('CUDA_Version', 'N/A')} \\
Total Model Parameters & {total_params:,} ({total_params/1e6:.2f}M) \\
Architecture Configuration & Blocks=[2, 4, 8, 16], Heads=[1, 2, 4, 8], Refinement=2 \\
Feed-Forward Network Expansion & $2.66\times$ Dual-Gated FFN \\
Training Dataset & LOL our485 (485 paired scenes) \\
Evaluation Dataset & LOL eval15 (15 standard test scenes) \\
Training Patch Size / Batch Size & {TRAIN_PATCH_SIZE} $\times$ {TRAIN_PATCH_SIZE} / {BATCH_SIZE} \\
Loss Objective & Smooth L1 Loss ($\mathcal{{L}}_1$) \\
Optimization Algorithm & Adam ($\beta_1=0.9, \beta_2=0.999, \epsilon=10^{{-8}}$) \\
Learning Rate Schedule & Cosine Annealing (${INITIAL_LR} \to {MIN_LR}$) \\
Training Duration / Epochs & 50 Epochs ({total_training_time/60:.1f} minutes) \\
Initial Checkpoint Status & None (Trained from Scratch, Epoch 0) \\
Best Achieved Validation PSNR & {best_val_psnr:.2f} dB (Epoch {checkpoint['epoch']}) \\
\hline
\end{{tabular}}
\end{{table}}
"""

with open(os.path.join(OUTPUT_DIR, 'table3_training_specifications.tex'), 'w') as f:
    f.write(tex_table3)

print("Exported all LaTeX tables:")
print(f"  1. {os.path.join(OUTPUT_DIR, 'table1_per_image_evaluation.tex')}")
print(f"  2. {os.path.join(OUTPUT_DIR, 'table2_benchmark_comparison.tex')}")
print(f"  3. {os.path.join(OUTPUT_DIR, 'table3_training_specifications.tex')}")


---
## Section 7: Final Paper Assets Inventory & Packaging
Summary of all generated files saved into `paper_output/` ready for inclusion in the IEEE conference report.


In [ ]:
# List and Package Generated Paper Deliverables
import shutil

print("=" * 75)
print("                   GENERATED PAPER ASSETS INVENTORY")
print("=" * 75)
exported_files = sorted(os.listdir(OUTPUT_DIR))
for f in exported_files:
    fpath = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fpath):
        size_kb = os.path.getsize(fpath) / 1024
        print(f"  [File] {f:<38} ({size_kb:>7.1f} KB)")
    else:
        print(f"  [Dir ] {f:<38}")
print("=" * 75)

# Create a zip archive of all paper assets for one-click download
zip_archive_path = os.path.join(DRIVE_DIR, 'LLFormer_Paper_Assets.zip')
shutil.make_archive(zip_archive_path.replace('.zip', ''), 'zip', OUTPUT_DIR)
print(f"\nAll paper assets packaged into: {zip_archive_path}")
print("Ready to import directly into Overleaf or your local LaTeX workspace!")
